In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("brijbhushannanda1979/bigmart-sales-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\manas\.cache\kagglehub\datasets\brijbhushannanda1979\bigmart-sales-data\versions\1


In [2]:
df = pd.read_csv("Train.csv")
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


In [5]:
# checking for missing values
df.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [4]:
df.drop(["Item_Identifier", "Outlet_Identifier"], axis=1, inplace=True)

In [5]:
df.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,9.30,Low Fat,0.016047,Dairy,249.8092,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,5.92,Regular,0.019278,Soft Drinks,48.2692,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,17.50,Low Fat,0.016760,Meat,141.6180,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,1998,NaN,Tier 3,Grocery Store,732.3800
4,8.93,Low Fat,0.000000,Household,53.8614,1987,High,Tier 3,Supermarket Type1,994.7052


In [7]:
df['Item_Type'].value_counts()

Item_Type
Fruits and Vegetables    1232
Snack Foods              1200
Household                 910
Frozen Foods              856
Dairy                     682
Canned                    649
Baking Goods              648
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64
Name: count, dtype: int64

In [9]:
# mean value of "Item_Weight" column
df['Item_Weight'].mean()

np.float64(12.857645184135977)

In [11]:
# filling the missing values in "Item_weight column" with "Mean" value
df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)

In [13]:
# mode of "Outlet_Size" column
df['Outlet_Size'].mode()

0    Medium
Name: Outlet_Size, dtype: object

In [15]:
# filling the missing values in "Outlet_Size" column with Mode
mode_of_Outlet_size = df.pivot_table(values='Outlet_Size', columns='Outlet_Type', aggfunc=(lambda x: x.mode()[0]))

In [16]:
df.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,9.30,Low Fat,0.016047,Dairy,249.8092,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,5.92,Regular,0.019278,Soft Drinks,48.2692,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,17.50,Low Fat,0.016760,Meat,141.6180,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,1998,NaN,Tier 3,Grocery Store,732.3800
4,8.93,Low Fat,0.000000,Household,53.8614,1987,High,Tier 3,Supermarket Type1,994.7052


In [17]:

miss_values = df['Outlet_Size'].isnull()   

In [18]:

print(miss_values)

0       False
1       False
2       False
3        True
4       False
        ...  
8518    False
8519     True
8520    False
8521    False
8522    False
Name: Outlet_Size, Length: 8523, dtype: bool


In [20]:
df.loc[miss_values, 'Outlet_Size'] = df.loc[miss_values,'Outlet_Type'].apply(lambda x: mode_of_Outlet_size[x])

In [22]:
# checking for missing values
df.isnull().sum()

Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

In [23]:
df.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
count,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,12.857645,0.066132,140.992782,1997.831867,2181.288914
std,4.226124,0.051598,62.275067,8.371760,1706.499616
min,4.555000,0.000000,31.290000,1985.000000,33.290000
25%,9.310000,0.026989,93.826500,1987.000000,834.247400
50%,12.857645,0.053931,143.012800,1999.000000,1794.331000
75%,16.000000,0.094585,185.643700,2004.000000,3101.296400
max,21.350000,0.328391,266.888400,2009.000000,13086.964800


In [24]:
df.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,9.30,Low Fat,0.016047,Dairy,249.8092,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,5.92,Regular,0.019278,Soft Drinks,48.2692,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,17.50,Low Fat,0.016760,Meat,141.6180,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,1998,Small,Tier 3,Grocery Store,732.3800
4,8.93,Low Fat,0.000000,Household,53.8614,1987,High,Tier 3,Supermarket Type1,994.7052


In [26]:
num_cols = [i for i in df.columns if df[i].dtype != "O"]
cat_cols = [i for i in df.columns if df[i].dtype == "O"]

num_cols , cat_cols

(['Item_Weight',
  'Item_Visibility',
  'Item_MRP',
  'Outlet_Establishment_Year',
  'Item_Outlet_Sales'],
 ['Item_Fat_Content',
  'Item_Type',
  'Outlet_Size',
  'Outlet_Location_Type',
  'Outlet_Type'])

In [27]:
df['Outlet_Type'].unique()

array(['Supermarket Type1', 'Supermarket Type2', 'Grocery Store',
       'Supermarket Type3'], dtype=object)

In [30]:
df['Outlet_Type'] = np.where(df['Outlet_Type'] == "Supermarket Type1", 1, df['Outlet_Type'])
df['Outlet_Type'] = np.where(df['Outlet_Type'] == "Supermarket Type2", 2, df['Outlet_Type'])
df['Outlet_Type'] = np.where(df['Outlet_Type'] == "Grocery Store", 3, df['Outlet_Type'])
df['Outlet_Type'] = np.where(df['Outlet_Type'] == "Supermarket Type3", 4, df['Outlet_Type'])


df['Outlet_Type'].unique()



array([1, 2, 3, 4], dtype=object)

In [31]:
df['Outlet_Location_Type'].value_counts()

Outlet_Location_Type
Tier 3    3350
Tier 2    2785
Tier 1    2388
Name: count, dtype: int64

In [32]:
df['Outlet_Location_Type'] = np.where(df['Outlet_Location_Type'] == "Tier 1", 1, df['Outlet_Location_Type'])
df['Outlet_Location_Type'] = np.where(df['Outlet_Location_Type'] == "Tier 2", 2, df['Outlet_Location_Type'])
df['Outlet_Location_Type'] = np.where(df['Outlet_Location_Type'] == "Tier 3", 3, df['Outlet_Location_Type'])

In [33]:
df['Outlet_Location_Type'].value_counts()

Outlet_Location_Type
3    3350
2    2785
1    2388
Name: count, dtype: int64

In [35]:
df['Outlet_Establishment_Year'] = pd.to_datetime(df['Outlet_Establishment_Year'])

In [36]:
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [50]:
df.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,9.30,1.0,0.016047,4.0,249.8092,2,1,1,3735.1380
1,5.92,2.0,0.019278,14.0,48.2692,2,3,2,443.4228
2,17.50,1.0,0.016760,10.0,141.6180,2,1,1,2097.2700
3,19.20,2.0,0.000000,6.0,182.0950,1,3,3,732.3800
4,8.93,1.0,0.000000,9.0,53.8614,3,3,1,994.7052


In [38]:
df.drop(["Outlet_Establishment_Year"], axis=1, inplace=True)

In [43]:
df['Outlet_Size'] = np.where(df['Outlet_Size'] == "Small", 1, df['Outlet_Size'])
df['Outlet_Size'] = np.where(df['Outlet_Size'] == "Medium", 2, df['Outlet_Size'])
df['Outlet_Size'] = np.where(df['Outlet_Size'] == "High", 3, df['Outlet_Size'])

In [44]:
df['Outlet_Size'].unique()

array([2, 1, 3], dtype=object)

In [46]:
df['Item_Fat_Content'].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [47]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder()

df['Item_Fat_Content'] = enc.fit_transform(df[['Item_Fat_Content']])


In [49]:
df['Item_Type'] = enc.fit_transform(df[['Item_Type']])

In [51]:
X = df.drop(columns='Item_Outlet_Sales', axis=1)
y = df['Item_Outlet_Sales']

from sklearn.model_selection import train_test_split 

X_train , X_test , y_train , y_test = train_test_split(X ,y ,test_size= 0.2 , random_state= 42)



In [52]:
X_train , y_test

(      Item_Weight  Item_Fat_Content  Item_Visibility  Item_Type  Item_MRP  \
 549         9.500               2.0         0.035206        6.0  171.3448   
 7757       18.000               1.0         0.047473        9.0  170.5422   
 764        17.600               2.0         0.076122       10.0  111.7202   
 6867        8.325               1.0         0.029845        6.0   41.6138   
 2716       12.850               1.0         0.137228       13.0  155.5630   
 ...           ...               ...              ...        ...       ...   
 5734        9.395               2.0         0.286345        6.0  139.1838   
 5191       15.600               1.0         0.117575        5.0   75.6670   
 5390       17.600               1.0         0.018944        8.0  237.3590   
 860        20.350               3.0         0.054363       13.0  117.9466   
 7270       16.350               0.0         0.016993        9.0   95.7410   
 
      Outlet_Size Outlet_Location_Type Outlet_Type  
 549     

In [53]:
from sklearn.svm import SVR 

regression = SVR(kernel = 'rbf' , C = 1 , gamma = 0.1 , epsilon= 0.1 , shrinking= True , verbose = True)

regression.fit(X_train , y_train)


[LibSVM]

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",True
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [54]:
y_pred = regression.predict(X_test)




In [55]:
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error

print("R2 Score : ", r2_score(y_test , y_pred))
print("Mean Squared Error : ", mean_squared_error(y_test , y_pred))
print("Mean Absolute Error : ", mean_absolute_error(y_test , y_pred))

R2 Score :  -0.02703596126568808
Mean Squared Error :  2791453.000059072
Mean Absolute Error :  1268.134051344176
